# 01 — Générer les CSV à partir de `MainTable.csv`

Ce notebook exécute les scripts Python présents dans `scripts/` afin de produire les fichiers de métriques dans le dossier `csv/`.

Organisation du projet :

```text
Chaine/
├── data/
│   └── MainTable.csv
├── scripts/
│   ├── compile_count.py
│   ├── compil_ratio.py
│   └── ...
└── csv/
```

Placez ce notebook à la racine du dossier `Chaine/`, ou ajustez `PROJECT_DIR` dans la cellule de configuration.

## 1. Configuration

Les paramètres ci-dessous peuvent être modifiés avant l’exécution.

In [ ]:
from pathlib import Path
import sys
import shutil
import os
import runpy
import contextlib
import traceback
import pandas as pd
from IPython.display import display

def detect_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "Chaine",
        Path.cwd() / "chaine_extracted" / "Chaine",
        Path.cwd().parent / "Chaine",
    ]
    for candidate in candidates:
        if (candidate / "data" / "MainTable.csv").exists() and (candidate / "scripts").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Impossible de détecter automatiquement le dossier Chaine. "
        "Modifiez PROJECT_DIR manuellement dans cette cellule."
    )

# Par défaut, le notebook détecte automatiquement le dossier projet.
# PROJECT_DIR = Path(r"C:/Users/.../Chaine")
PROJECT_DIR = detect_project_dir()

DATA_DIR = PROJECT_DIR / "data"
MAIN_TABLE_PATH = DATA_DIR / "MainTable.csv"
SCRIPTS_DIR = PROJECT_DIR / "scripts"
CSV_DIR = PROJECT_DIR / "csv"
LOG_DIR = CSV_DIR / "logs"

# Si True, supprime uniquement les CSV produits par les scripts listés ci-dessous avant de les régénérer.
OVERWRITE_OUTPUTS = True

# Si True, supprime le cache créé par data_filter.py pour forcer un recalcul sur MainTable.csv.
CLEAR_DATA_FILTER_CACHE = False

CSV_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Projet :", PROJECT_DIR)
print("MainTable :", MAIN_TABLE_PATH)
print("Scripts :", SCRIPTS_DIR)
print("Sorties CSV :", CSV_DIR)

Projet : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine
MainTable : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine\data\MainTable.csv
Scripts : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine\scripts
Sorties CSV : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine\csv


## 2. Vérification du fichier d’entrée

In [ ]:
if not MAIN_TABLE_PATH.exists():
    raise FileNotFoundError(f"Fichier introuvable : {MAIN_TABLE_PATH}")

if not SCRIPTS_DIR.exists():
    raise FileNotFoundError(f"Dossier scripts introuvable : {SCRIPTS_DIR}")

main_table = pd.read_csv(MAIN_TABLE_PATH)
print(f"MainTable chargé : {main_table.shape[0]:,} lignes × {main_table.shape[1]:,} colonnes")
display(pd.DataFrame({
    "colonne": main_table.columns,
    "non_null": [main_table[c].notna().sum() for c in main_table.columns],
    "dtype": [str(main_table[c].dtype) for c in main_table.columns],
}))

MainTable chargé : 5,710 lignes × 17 colonnes


,colonne,non_null,dtype
0,SubjectID,5710,object
1,ToolInstances,5710,object
2,ServerTimestamp,5710,object
3,ServerTimezone,5710,int64
4,CourseID,5710,object
5,AssignmentID,5710,int64
6,ProblemID,5710,int64
7,CodeStateID,5710,object
8,IsEventOrderingConsistent,5710,bool
9,EventType,5710,object


## 3. Définition des scripts à exécuter


In [ ]:
SCRIPT_JOBS = [
    {
        "script": "compile_count.py",
        "output": "CompileCount.csv",
        "description": "Nombre de compilations",
        "required": ["SubjectID", "EventType"],
    },
    {
        "script": "compil_ratio.py",
        "output": "CompileSuccessRate.csv",
        "description": "Taux de compilations réussies",
        "required": ["SubjectID", "EventType", "Compile.Result"],
    },
    {
        "script": "compile_span.py",
        "output": "CompileSpan.csv",
        "description": "Durée entre première et dernière compilation, en minutes",
        "required": ["SubjectID", "EventType", "ServerTimestamp"],
    },
    {
        "script": "eq.py",
        "output": "ErrorQuotient.csv",
        "description": "Error Quotient",
        "required": ["SubjectID", "Order", "EventType", "EventID", "ParentEventID", "CompileMessageType"],
    },
    {
        "script": "red.py",
        "output": "RED.csv",
        "description": "Repeated Error Density",
        "required": ["SubjectID", "Order", "EventType", "EventID", "ParentEventID", "CompileMessageType"],
    },
    {
        "script": "watwin.py",
        "output": "Watwin.csv",
        "description": "WatWin",
        "required": [
            "SubjectID", "Order", "EventType", "EventID", "CodeStateID",
            "ParentEventID", "CompileMessageData", "CompileMessageType",
            "SourceLocation"
        ],
        # Le script accepte ServerTimestamp ou ClientTimestamp.
        "any_required": [["ServerTimestamp", "ClientTimestamp"]],
    },
    {
        "script": "session_count.py",
        "output": "SessionCount.csv",
        "description": "Nombre de sessions par étudiant",
        "required": ["SubjectID", "EventType", "ServerTimestamp"],
        "extra_args": ["5.0"],
    },
    {
        "script": "frac_long.py",
        "output": "FracLong.csv",
        "description": "Fraction de pauses longues entre compilations",
        "required": ["SubjectID", "EventType", "ServerTimestamp"],
        "extra_args": ["5.0"],
    },
    {
        "script": "mean_test_score.py",
        "output": "MeanTestScore.csv",
        "description": "Score moyen des tests",
        "required": ["SubjectID", "EventType", "Score"],
    },
    {
        "script": "first_compile_vs_global_first.py",
        "output": "FirstCompileVsGlobalFirst.csv",
        "description": "Minutes entre la première compilation globale et la première compilation de l'étudiant",
        "required": ["SubjectID", "EventType", "ServerTimestamp"],
    },
    {
        "script": "last_compile_vs_global_last.py",
        "output": "LastCompileVsGlobalLast.csv",
        "description": "Minutes entre la dernière compilation de l'étudiant et la dernière compilation globale",
        "required": ["SubjectID", "EventType", "ServerTimestamp"],
    },
    {
        "script": "time_to_score1.py",
        "output": "TimeToScore1.csv",
        "description": "Temps jusqu'au premier score égal à 1",
        "required": ["SubjectID", "EventType", "ServerTimestamp", "Score"],
    },
]

jobs_df = pd.DataFrame([{k: v for k, v in job.items() if k not in {"required", "any_required", "extra_args"}} for job in SCRIPT_JOBS])
display(jobs_df)

,script,output,description
0,compile_count.py,CompileCount.csv,Nombre de compilations
1,compil_ratio.py,CompileSuccessRate.csv,Taux de compilations réussies
2,compile_span.py,CompileSpan.csv,"Durée entre première et dernière compilation, ..."
3,eq.py,ErrorQuotient.csv,Error Quotient
4,red.py,RED.csv,Repeated Error Density
5,watwin.py,Watwin.csv,WatWin
6,session_count.py,SessionCount.csv,Nombre de sessions par étudiant
7,frac_long.py,FracLong.csv,Fraction de pauses longues entre compilations
8,mean_test_score.py,MeanTestScore.csv,Score moyen des tests
9,first_compile_vs_global_first.py,FirstCompileVsGlobalFirst.csv,Minutes entre la première compilation globale ...


## 4. Exécution des scripts

In [ ]:
def missing_columns(job, columns):
    missing = [c for c in job.get("required", []) if c not in columns]
    for group in job.get("any_required", []):
        if not any(c in columns for c in group):
            missing.append(" ou ".join(group))
    return missing

def output_summary(path: Path):
    if not path.exists():
        return {"rows": None, "columns": None}
    try:
        df = pd.read_csv(path)
        return {
            "rows": int(df.shape[0]),
            "columns": ", ".join(df.columns.astype(str).tolist()),
        }
    except Exception as exc:
        return {"rows": None, "columns": f"lecture impossible : {exc}"}

@contextlib.contextmanager
def temporary_cwd(path: Path):
    old_cwd = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old_cwd)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

if CLEAR_DATA_FILTER_CACHE:
    cache_dir = PROJECT_DIR / "cache"
    if cache_dir.exists():
        shutil.rmtree(cache_dir)
        print(f"Cache supprimé : {cache_dir}")

report = []
available_columns = set(main_table.columns)

for job in SCRIPT_JOBS:
    print(f"Lancement : {job['script']} → {job['output']}", flush=True)

    script_path = SCRIPTS_DIR / job["script"]
    output_path = CSV_DIR / job["output"]
    log_path = LOG_DIR / f"{Path(job['script']).stem}.log"

    row = {
        "script": job["script"],
        "output": job["output"],
        "description": job["description"],
        "status": None,
        "returncode": None,
        "output_path": str(output_path),
        "rows": None,
        "columns": None,
        "log_path": str(log_path),
        "message": "",
    }

    if not script_path.exists():
        row.update(status="skipped", message=f"Script introuvable : {script_path}")
        report.append(row)
        print(f"Terminé : {job['script']} — {row['status']} ({row['message']})", flush=True)
        continue

    missing = missing_columns(job, available_columns)
    if missing:
        row.update(status="skipped_missing_columns", message=f"Colonnes manquantes : {missing}")
        report.append(row)
        print(f"Terminé : {job['script']} — {row['status']} ({row['message']})", flush=True)
        continue

    if OVERWRITE_OUTPUTS and output_path.exists():
        output_path.unlink()

    argv = [
        str(script_path),
        str(DATA_DIR),
        str(output_path),
        *job.get("extra_args", []),
    ]

    old_argv = sys.argv[:]
    try:
        with log_path.open("w", encoding="utf-8") as log_file:
            log_file.write("COMMANDE LOGIQUE\npython " + " ".join(argv) + "\n\nSORTIE DU SCRIPT\n")
            log_file.flush()
            with temporary_cwd(PROJECT_DIR), contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                sys.argv = argv
                try:
                    runpy.run_path(str(script_path), run_name="__main__")
                    returncode = 0
                except SystemExit as exc:
                    returncode = int(exc.code or 0) if isinstance(exc.code, int) else 1
        summary = output_summary(output_path)
        status = "ok" if returncode == 0 and output_path.exists() else "error"
        message = "CSV produit" if output_path.exists() else "Aucun CSV produit"
    except Exception as exc:
        returncode = 1
        status = "error"
        message = f"{type(exc).__name__}: {exc}"
        with log_path.open("a", encoding="utf-8") as log_file:
            log_file.write("\n\nEXCEPTION NOTEBOOK\n")
            traceback.print_exc(file=log_file)
        summary = output_summary(output_path)
    finally:
        sys.argv = old_argv

    row.update(
        status=status,
        returncode=returncode,
        rows=summary["rows"],
        columns=summary["columns"],
        message=message,
    )
    print(f"Terminé : {job['script']} — {row['status']} ({row['message']})", flush=True)
    report.append(row)

report_df = pd.DataFrame(report)
report_path = LOG_DIR / "run_report.csv"
report_df.to_csv(report_path, index=False)

display(report_df)
print(f"Rapport sauvegardé : {report_path}")

Lancement : compile_count.py → CompileCount.csv
Terminé : compile_count.py — ok (CSV produit)
Lancement : compil_ratio.py → CompileSuccessRate.csv
Terminé : compil_ratio.py — ok (CSV produit)
Lancement : compile_span.py → CompileSpan.csv
Terminé : compile_span.py — ok (CSV produit)
Lancement : eq.py → ErrorQuotient.csv
Terminé : eq.py — ok (CSV produit)
Lancement : red.py → RED.csv
Terminé : red.py — ok (CSV produit)
Lancement : watwin.py → Watwin.csv
Terminé : watwin.py — skipped_missing_columns (Colonnes manquantes : ['SourceLocation'])
Lancement : session_count.py → SessionCount.csv
Terminé : session_count.py — ok (CSV produit)
Lancement : frac_long.py → FracLong.csv
Terminé : frac_long.py — ok (CSV produit)
Lancement : mean_test_score.py → MeanTestScore.csv
Terminé : mean_test_score.py — ok (CSV produit)
Lancement : first_compile_vs_global_first.py → FirstCompileVsGlobalFirst.csv
Terminé : first_compile_vs_global_first.py — ok (CSV produit)
Lancement : last_compile_vs_global_last.p

,script,output,description,status,returncode,output_path,rows,columns,log_path,message
0,compile_count.py,CompileCount.csv,Nombre de compilations,ok,0.0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,110.0,"SubjectID, CompileCount",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
1,compil_ratio.py,CompileSuccessRate.csv,Taux de compilations réussies,ok,0.0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,110.0,"SubjectID, CompileSuccessRate",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
2,compile_span.py,CompileSpan.csv,"Durée entre première et dernière compilation, ...",ok,0.0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,110.0,"SubjectID, CompileSpanMinutes",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
3,eq.py,ErrorQuotient.csv,Error Quotient,ok,0.0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,110.0,"SubjectID, ErrorQuotient",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
4,red.py,RED.csv,Repeated Error Density,ok,0.0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,110.0,"SubjectID, RED",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
5,watwin.py,Watwin.csv,WatWin,skipped_missing_columns,NaN,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,NaN,None,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,Colonnes manquantes : ['SourceLocation']
6,session_count.py,SessionCount.csv,Nombre de sessions par étudiant,ok,0.0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,110.0,"SubjectID, SessionCount",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
7,frac_long.py,FracLong.csv,Fraction de pauses longues entre compilations,ok,0.0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,110.0,"SubjectID, FracLong",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
8,mean_test_score.py,MeanTestScore.csv,Score moyen des tests,ok,0.0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,110.0,"SubjectID, MeanTestScore",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit
9,first_compile_vs_global_first.py,FirstCompileVsGlobalFirst.csv,Minutes entre la première compilation globale ...,ok,0.0,C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,110.0,"SubjectID, MinutesFromGlobalFirstCompile",C:\Users\chris\Desktop\Code\TracesPresenter\Tr...,CSV produit


Rapport sauvegardé : C:\Users\chris\Desktop\Code\TracesPresenter\Traces_presenter\Jupyter\Chaine\csv\logs\run_report.csv


## 5. Contrôle des sorties

In [ ]:
generated = []
for csv_path in sorted(CSV_DIR.glob("*.csv")):
    try:
        df = pd.read_csv(csv_path)
        generated.append({
            "fichier": csv_path.name,
            "lignes": df.shape[0],
            "colonnes": df.shape[1],
            "noms_colonnes": ", ".join(df.columns.astype(str).tolist()),
        })
    except Exception as exc:
        generated.append({
            "fichier": csv_path.name,
            "lignes": None,
            "colonnes": None,
            "noms_colonnes": f"lecture impossible : {exc}",
        })

generated_df = pd.DataFrame(generated)
display(generated_df)

,fichier,lignes,colonnes,noms_colonnes
0,CompileCount.csv,110,2,"SubjectID, CompileCount"
1,CompileSpan.csv,110,2,"SubjectID, CompileSpanMinutes"
2,CompileSuccessRate.csv,110,2,"SubjectID, CompileSuccessRate"
3,ErrorQuotient.csv,110,2,"SubjectID, ErrorQuotient"
4,FirstCompileVsGlobalFirst.csv,110,2,"SubjectID, MinutesFromGlobalFirstCompile"
5,FracLong.csv,110,2,"SubjectID, FracLong"
6,LastCompileVsGlobalLast.csv,110,2,"SubjectID, MinutesToGlobalLastCompile"
7,MeanTestScore.csv,110,2,"SubjectID, MeanTestScore"
8,RED.csv,110,2,"SubjectID, RED"
9,SessionCount.csv,110,2,"SubjectID, SessionCount"


## 6. Affichage des journaux des erreurs éventuelles

In [ ]:
error_rows = report_df[report_df["status"].isin(["error", "skipped_missing_columns", "skipped"])].copy()

if error_rows.empty:
    print("Aucune erreur détectée.")
else:
    display(error_rows[["script", "status", "message", "log_path"]])
    for _, row in error_rows.iterrows():
        log_path = Path(row["log_path"])
        print("\n" + "=" * 100)
        print(f"{row['script']} — {row['status']}")
        print(row["message"])
        if log_path.exists():
            log_text = log_path.read_text(encoding="utf-8", errors="replace")
            print(log_text[-4000:])

,script,status,message,log_path
5,watwin.py,skipped_missing_columns,Colonnes manquantes : ['SourceLocation'],C:\Users\chris\Desktop\Code\TracesPresenter\Tr...



watwin.py — skipped_missing_columns
Colonnes manquantes : ['SourceLocation']
